In [ ]:
# 1. Safe Library Installation (Uses active environment cache)
%pip install -q transformers accelerate datasets

# 2. Prevent HuggingFace from flooding your 20GB disk space
import os
os.environ["HF_HOME"] = "/kaggle/working/hf_cache"

# 3. Hardware Verification Setup
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 System active. Current Device: {device.upper()}")
print(f"📊 Total available GPUs detected: {torch.cuda.device_count()}")

if torch.cuda.device_count() > 1:
    print("💡 Pro-Tip: You are running dual T4 GPUs. Use torch.nn.DataParallel() to maximize VRAM!")

Note: you may need to restart the kernel to use updated packages.
🚀 System active. Current Device: CPU
📊 Total available GPUs detected: 0


c:\Users\ADEGOKE\Desktop\DS-ML-AI\E-Commerce Risk And Demand Intelligent System\.kaggle-env\Scripts\python.exe: No module named pip


: 

In [ ]:
## Kaggle Cloud Path


from pathlib import Path
import pandas as pd

input_root = Path("/kaggle/input")
target_file = "nigeria_ecommerce_major_project_25000.csv"

if not input_root.exists():
    raise RuntimeError(
        "This cell must run on Kaggle, not the local VS Code kernel. "
        "Push the notebook with 'kaggle kernels push -p .' and run it on Kaggle."
    )

dataset_path = next(input_root.rglob(target_file), None)

if dataset_path is None:
    raise FileNotFoundError(
        f"{target_file} was not found under {input_root}. "
        "Confirm that the Kaggle dataset is attached to this Kernel."
    )

print(f"Using Kaggle dataset: {dataset_path}")
df = pd.read_csv(dataset_path)
df.head()

In [1]:
# Imports and reproducibility setup
from pathlib import Path
import json
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
warnings.filterwarnings("ignore")



df = pd.read_csv('../data/nigeria_ecommerce_25000.csv')

print("Shape:", df.shape)
display(df.head())

Shape: (25000, 31)


,order_id,customer_id,order_timestamp,order_date,city,customer_type,customer_tenure_days,previous_orders,prior_cancellations,email_verified,...,shipping_fee,delivery_distance_km,estimated_delivery_days,payment_method,gross_amount,discount_amount,total_amount,order_status,actual_delivery_days,customer_rating
0,1,13681,2024-03-06 05:49:07,2024-03-06,Ilorin,New,6,0,0,True,...,1900.0,25.4,4,Card,8500.0,850.0,9550.0,Delivered,5.0,4.0
1,2,12075,2025-07-19 18:17:04,2025-07-19,Benin City,Returning,402,14,0,True,...,1900.0,25.4,5,Card,264800.0,26480.0,240220.0,Delivered,7.0,4.0
2,3,10402,2025-04-23 11:48:09,2025-04-23,Abuja,Returning,221,5,0,True,...,2500.0,38.3,5,Bank Transfer,54600.0,0.0,57100.0,Delivered,6.0,4.0
3,4,10764,2024-11-16 19:40:59,2024-11-16,Port Harcourt,Returning,178,13,1,False,...,2100.0,28.6,5,Cash on Delivery,238800.0,11940.0,228960.0,Delivered,6.0,3.0
4,5,11214,2024-11-12 12:49:09,2024-11-12,Port Harcourt,New,82,0,0,True,...,2400.0,34.9,5,Bank Transfer,11600.0,1160.0,12840.0,Cancelled,NaN,NaN


In [2]:
# 1. Number of rows
print("Number of rows:", len(df))

# 2. Number of unique order IDs
print("Unique order IDs:", df["order_id"].nunique())

# 3. Are order IDs duplicated?
print("Duplicated order IDs:", df["order_id"].duplicated().sum())

# 4. Show duplicated orders if any
duplicated_orders = df[df["order_id"].duplicated(keep=False)].sort_values("order_id")

duplicated_orders.head(20)

Number of rows: 25000
Unique order IDs: 25000
Duplicated order IDs: 0


,order_id,customer_id,order_timestamp,order_date,city,customer_type,customer_tenure_days,previous_orders,prior_cancellations,email_verified,...,shipping_fee,delivery_distance_km,estimated_delivery_days,payment_method,gross_amount,discount_amount,total_amount,order_status,actual_delivery_days,customer_rating


In [3]:
df["order_status"].value_counts(dropna=False)

order_status
Delivered    16649
Cancelled     7659
Refunded       692
Name: count, dtype: int64

In [4]:
df["order_status"].value_counts(normalize=True, dropna=False) * 100

order_status
Delivered    66.596
Cancelled    30.636
Refunded      2.768
Name: proportion, dtype: float64

In [5]:
pd.crosstab(
    df["order_status"],
    columns="count"
)

col_0,count
order_status,
Cancelled,7659
Delivered,16649
Refunded,692


In [6]:
pd.crosstab(
    df["order_status"],
    df["payment_method"],
    normalize="index"
).round(3)

payment_method,Bank Transfer,Card,Cash on Delivery,Wallet
order_status,,,,
Cancelled,0.278,0.339,0.209,0.174
Delivered,0.305,0.395,0.108,0.192
Refunded,0.293,0.396,0.124,0.186


In [7]:
pd.crosstab(
    df["order_status"],
    df["payment_method"]
)

payment_method,Bank Transfer,Card,Cash on Delivery,Wallet
order_status,,,,
Cancelled,2129,2599,1600,1331
Delivered,5084,6569,1794,3202
Refunded,203,274,86,129


In [8]:
pd.crosstab(
    df["order_status"],
    df["category"],
    normalize="index"
).round(3)

category,Beauty & Personal Care,Electronics,Fashion,Home & Kitchen,Sports & Fitness
order_status,,,,,
Cancelled,0.148,0.243,0.225,0.201,0.184
Delivered,0.154,0.245,0.222,0.200,0.178
Refunded,0.104,0.361,0.165,0.233,0.137


In [9]:
df.groupby("order_status").agg(
    orders=("order_id", "count"),
    customers=("customer_id", "nunique"),
    avg_quantity=("quantity", "mean"),
    avg_unit_price=("unit_price", "mean"),
    avg_discount=("discount_percent", "mean"),
    avg_total_amount=("total_amount", "mean")
).round(2)

,orders,customers,avg_quantity,avg_unit_price,avg_discount,avg_total_amount
order_status,,,,,,
Cancelled,7659,3903,2.45,28996.87,10.16,64689.32
Delivered,16649,4811,2.45,28851.47,9.63,64661.43
Refunded,692,651,2.45,31588.15,9.92,68720.06


In [10]:
df.groupby("order_status")["total_amount"].describe().round(2)

,count,mean,std,min,25%,50%,75%,max
order_status,,,,,,,,
Cancelled,7659.0,64689.32,50197.03,6390.0,29160.0,50500.0,83825.0,553300.0
Delivered,16649.0,64661.43,50579.54,6540.0,29450.0,50195.0,83800.0,612400.0
Refunded,692.0,68720.06,52078.31,8670.0,32080.0,55440.0,91182.5,467120.0


In [11]:
df.groupby("order_status")["discount_percent"].describe().round(2)

,count,mean,std,min,25%,50%,75%,max
order_status,,,,,,,,
Cancelled,7659.0,10.16,8.97,0.0,0.0,10.0,15.0,30.0
Delivered,16649.0,9.63,8.60,0.0,0.0,10.0,15.0,30.0
Refunded,692.0,9.92,8.60,0.0,0.0,10.0,15.0,30.0


In [12]:
pd.crosstab(
    df["order_status"],
    df["city"],
    normalize="index"
).round(3)

city,Abuja,Benin City,Enugu,Ibadan,Ilorin,Kano,Lagos,Port Harcourt
order_status,,,,,,,,
Cancelled,0.155,0.091,0.075,0.135,0.074,0.107,0.265,0.098
Delivered,0.160,0.076,0.068,0.114,0.068,0.099,0.312,0.103
Refunded,0.142,0.082,0.066,0.114,0.082,0.084,0.321,0.108


In [13]:
pd.crosstab(
    df["order_status"],
    df["category"],
    normalize="index"
).round(3)

category,Beauty & Personal Care,Electronics,Fashion,Home & Kitchen,Sports & Fitness
order_status,,,,,
Cancelled,0.148,0.243,0.225,0.201,0.184
Delivered,0.154,0.245,0.222,0.200,0.178
Refunded,0.104,0.361,0.165,0.233,0.137


In [14]:
df["order_date"] = pd.to_datetime(df["order_date"])

print(df["order_date"].min())
print(df["order_date"].max())

2024-01-01 00:00:00
2025-12-31 00:00:00


In [15]:
df.groupby(
    df["order_date"].dt.to_period("M")
)["order_status"].value_counts().unstack(fill_value=0)

order_status,Cancelled,Delivered,Refunded
order_date,,,
2024-01,301,739,45
2024-02,318,647,20
2024-03,323,697,21
2024-04,314,712,24
2024-05,332,705,35
2024-06,320,685,30
2024-07,307,703,34
2024-08,309,744,34
2024-09,341,655,30


In [16]:
status_by_month = pd.crosstab(
    df["order_date"].dt.to_period("M"),
    df["order_status"],
    normalize="index"
) * 100

status_by_month.round(2)

order_status,Cancelled,Delivered,Refunded
order_date,,,
2024-01,27.74,68.11,4.15
2024-02,32.28,65.69,2.03
2024-03,31.03,66.95,2.02
2024-04,29.90,67.81,2.29
2024-05,30.97,65.76,3.26
2024-06,30.92,66.18,2.90
2024-07,29.41,67.34,3.26
2024-08,28.43,68.45,3.13
2024-09,33.24,63.84,2.92


In [17]:
expected_total = (
    df["quantity"]
    * df["unit_price"]
    * (1 - df["discount_percent"] / 100)
)

df["total_difference"] = (
    df["total_amount"] - expected_total
)

df["total_difference"].describe()

count    25000.000000
mean      1910.984000
std        376.874824
min        800.000000
25%       1700.000000
50%       1900.000000
75%       2200.000000
max       3600.000000
Name: total_difference, dtype: float64

In [18]:
df[["quantity",
    "unit_price",
    "discount_percent",
    "total_amount",
    "total_difference"]].head(20)

,quantity,unit_price,discount_percent,total_amount,total_difference
0,1,8500.0,10,9550.0,1900.0
1,4,66200.0,10,240220.0,1900.0
2,1,54600.0,0,57100.0,2500.0
3,3,79600.0,5,228960.0,2100.0
4,1,11600.0,10,12840.0,2400.0
5,1,15800.0,20,14940.0,2300.0
6,3,12300.0,25,29375.0,1700.0
7,2,8600.0,20,15960.0,2200.0
8,2,11500.0,5,23950.0,2100.0
9,1,23100.0,25,19125.0,1800.0


In [19]:
print(
    "Rows matching formula:",
    (df["total_difference"].abs() < 0.01).sum()
)

print(
    "Total rows:",
    len(df)
)

Rows matching formula: 0
Total rows: 25000


In [20]:
print(df.columns.tolist())

['order_id', 'customer_id', 'order_timestamp', 'order_date', 'city', 'customer_type', 'customer_tenure_days', 'previous_orders', 'prior_cancellations', 'email_verified', 'traffic_source', 'device_type', 'session_duration_minutes', 'pages_viewed', 'cart_items', 'product', 'category', 'quantity', 'unit_price', 'discount_percent', 'coupon_code', 'shipping_fee', 'delivery_distance_km', 'estimated_delivery_days', 'payment_method', 'gross_amount', 'discount_amount', 'total_amount', 'order_status', 'actual_delivery_days', 'customer_rating', 'total_difference']


In [21]:
expected_total = (
    df["gross_amount"]
    - df["discount_amount"]
    + df["shipping_fee"]
)

difference = df["total_amount"] - expected_total

print("Rows matching formula:", (difference.abs() < 0.01).sum())
print("Total rows:", len(df))

print("\nDifference summary:")
print(difference.describe())

print("\nRows that do NOT match:")
display(
    df.loc[
        difference.abs() >= 0.01,
        [
            "quantity",
            "unit_price",
            "gross_amount",
            "discount_percent",
            "discount_amount",
            "shipping_fee",
            "total_amount"
        ]
    ].head(20)
)

Rows matching formula: 25000
Total rows: 25000

Difference summary:
count    25000.0
mean         0.0
std          0.0
min          0.0
25%          0.0
50%          0.0
75%          0.0
max          0.0
dtype: float64

Rows that do NOT match:


,quantity,unit_price,gross_amount,discount_percent,discount_amount,shipping_fee,total_amount
